In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install -q chemprop

import pandas as pd, numpy as np, subprocess, shutil, glob, torch
pd.set_option("display.width", 200)
print("GPU available:", torch.cuda.is_available())

data_file = glob.glob("/kaggle/input/**/my_data_clean.csv", recursive=True)[0]
clean = pd.read_csv(data_file)
clean["abs_eV"] = 1239.84 / clean.abs_nm          # absorption as energy; NaN where not measured
print(clean.shape, "| absorption available:", clean.abs_eV.notna().sum())

GPU available: True
(17850, 10) | absorption available: 15758


In [2]:
def make_split(df, seed):
    groups = df.groupby("scaffold").size().sample(frac=1, random_state=seed)
    big = groups[groups > len(df) * 0.005].index.tolist()
    small = [g for g in groups.index if g not in set(big)]
    split, count, n = {}, 0, len(df)
    for g in big + small:
        split[g] = "train" if count < 0.8 * n else ("val" if count < 0.9 * n else "test")
        count += groups[g]
    return df.scaffold.map(split)

SEEDS = [42, 0, 1, 2, 3]
work = "/tmp/mt_work"
out = "/kaggle/working/multitask"
os.makedirs(work, exist_ok=True); os.makedirs(out, exist_ok=True)

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise RuntimeError("command failed: " + cmd[:80])

In [3]:
for s in SEEDS:
    if os.path.exists(f"{out}/preds_mt_seed{s}.csv"):
        print("already done: seed", s)
        continue
    d = clean.assign(split=make_split(clean, s))
    d[["chrom", "solv", "emi_eV", "abs_eV", "split"]].to_csv(f"{work}/all.csv", index=False)
    d[d.split == "test"][["chrom", "solv"]].to_csv(f"{work}/test.csv", index=False)   # SMILES only

    run(f"rm -rf {work}/m_mt")
    run(f"chemprop train --data-path {work}/all.csv --splits-column split "
        f"--smiles-columns chrom solv --target-columns emi_eV abs_eV --task-type regression "
        f"--epochs 40 --patience 10 --batch-size 64 --depth 4 --message-hidden-dim 300 --ffn-num-layers 2 "
        f"--pytorch-seed {s} --save-dir {work}/m_mt")
    run(f"chemprop predict --test-path {work}/test.csv --smiles-columns chrom solv "
        f"--model-paths {work}/m_mt --preds-path {out}/preds_mt_seed{s}.csv")
    shutil.copy(f"{work}/m_mt/model_0/best.pt", f"{out}/mt_seed{s}.pt")
    print("seed", s, "done")

seed 42 done
seed 0 done
seed 1 done
seed 2 done
seed 3 done


In [4]:
def metrics(true, pred):
    err = pred - true
    hit = pred >= 700
    r = {"MAE all": np.abs(err).mean()}
    for lab, lo, hi in [("<500", 0, 500), ("500-650", 500, 650), ("650-700", 650, 700), (">=700", 700, 2000)]:
        m = (true >= lo) & (true < hi)
        r[f"MAE {lab}"] = np.abs(err[m]).mean()
    r["bias >=700"] = err[true >= 700].mean()
    r["recall %"] = 100 * hit[true >= 700].mean()
    r["false hits"] = int((hit & (true < 650)).sum())
    r["precision %"] = 100 * (true[hit] >= 700).mean() if hit.any() else np.nan
    return r

rows = []
for s in SEEDS:
    t = clean[make_split(clean, s) == "test"]
    p = pd.read_csv(f"{out}/preds_mt_seed{s}.csv")
    emi_pred = 1239.84 / p["emi_eV"].values           # read the emission column by name
    abs_pred = 1239.84 / p["abs_eV"].values
    r = {"seed": s, **metrics(t.emi_nm.values, emi_pred)}
    has_abs = t.abs_nm.notna().values
    r["absorption MAE"] = np.abs(abs_pred[has_abs] - t.abs_nm.values[has_abs]).mean()   # side result
    rows.append(r)

res = pd.DataFrame(rows)
res.round(1).to_csv(f"{out}/multitask_results.csv", index=False)
print(res.round(1).to_string(index=False))
print("\nmean:\n", res.drop(columns="seed").mean().round(1).to_string())

 seed  MAE all  MAE <500  MAE 500-650  MAE 650-700  MAE >=700  bias >=700  recall %  false hits  precision %  absorption MAE
   42     31.5      27.0         34.7         36.9       54.7       -37.3      68.8           3         82.8            26.9
    0     32.7      27.1         40.3         36.8       52.6       -32.9      66.7           0         87.8            24.0
    1     30.5      26.6         35.1         42.3       62.7       -35.8      56.2           2         64.3            22.3
    2     35.8      26.5         41.2         55.4       48.3       -37.2      47.0          19         66.1            28.8
    3     33.7      30.8         37.7         38.9       25.5       -14.5      69.0           1         81.6            26.9

mean:
 MAE all           32.8
MAE <500          27.6
MAE 500-650       37.8
MAE 650-700       42.1
MAE >=700         48.7
bias >=700       -31.5
recall %          61.5
false hits         5.0
precision %       76.5
absorption MAE    25.8


In [5]:
!cd /kaggle/working && zip -qr multitask_results.zip multitask && ls -lh multitask_results.zip

-rw-r--r-- 1 root root 14M Oct 25 15:23 multitask_results.zip
